# Publications markdown generator for academicpages

Takes a TSV of publications with metadata and converts them for use with [academicpages.github.io](academicpages.github.io). This is an interactive Jupyter notebook ([see more info here](http://jupyter-notebook-beginner-guide.readthedocs.io/en/latest/what_is_jupyter.html)). The core python code is also in `publications.py`. Run either from the `markdown_generator` folder after replacing `publications.tsv` with one containing your data.

TODO: Make this work with BibTex and other databases of citations, rather than Stuart's non-standard TSV format and citation style.


## Data format

The TSV needs to have the following columns: pub_date, title, venue, excerpt, citation, site_url, and paper_url, with a header at the top. 

- `excerpt` and `paper_url` can be blank, but the others must have values. 
- `pub_date` must be formatted as YYYY-MM-DD.
- `url_slug` will be the descriptive part of the .md file and the permalink URL for the page about the paper. The .md file will be `YYYY-MM-DD-[url_slug].md` and the permalink will be `https://[yourdomain]/publications/YYYY-MM-DD-[url_slug]`

This is how the raw file looks (it doesn't look pretty, use a spreadsheet or other program to edit and create).

In [17]:
!cat publications.tsv

'cat' is not recognized as an internal or external command,
operable program or batch file.


## Import pandas

We are using the very handy pandas library for dataframes.

In [18]:
import pandas as pd

## Import TSV

Pandas makes this easy with the read_csv function. We are using a TSV, so we specify the separator as a tab, or `\t`.

I found it important to put this data in a tab-separated values format, because there are a lot of commas in this kind of data and comma-separated values can get messed up. However, you can modify the import statement, as pandas also has read_excel(), read_json(), and others.

In [19]:
publications = pd.read_csv("publications.tsv", sep="\t", header=0)
publications


,pub_date,title,venue,excerpt,citation,url_slug,paper_url
0,2022,Landscapes in Motion: Cartographies of Connect...,Routledge Handbook of the Digital Environmenta...,In this paper we argue for leveraging geospati...,"Horne, Ryan and Ruth Mostern. “Landscapes in ...",landscapes-in-motion,NaN
1,2021,Linked Open Data for the Ancient Mediterranea...,ISAW Papers,This collection presents case studies written ...,"Bond, Sarah E., Paul C. Dilley, and Ryan Horne...",isaw-20,http://hdl.handle.net/2333.1/gqnk9kz2.
2,2020,"Aeolian Alexanders: Economic, Material, and So...",GitHub,NEH funded project that reimagines die studies...,"Horne, Ryan. (220). ""Aeolian Alexanders.""",aeolian-alexanders,https://github.com/Aeolian-Alexanders
3,2021,Digital Approaches to the ‘Big Ancient Mediter...,Access and Control in Digital Humanities,This article serves as an introduction to BAM ...,"Horne, Ryan. “Digital Approaches to the ‘Big A...",digital-approaches-to-bam,https://www.taylorfrancis.com/chapters/edit/10...
4,2021,Introducing the Semantic Web and Linked Open Data,ISAW Papers,This introduction is a series of introductory ...,"Bond, Sarah E., Paul C. Dilley, and Ryan Horne...",isaw-20-intro,http://hdl.handle.net/2333.1/c2fqzh3d.
5,2021,Applying Linked Open Data Standards,ISAW Papers,This chapter serves as an introduction to the ...,"Horne, Ryan. “Applying Linked Open Data Standa...",isaw-20-applying-lod,http://hdl.handle.net/2333.1/79cnpgk2.
6,2021,Tracking Yu: Developing a Data System for Yell...,Yellow River: A Natural and Unnatural History,An overview of the use of digital humanities t...,"Mostern, Ruth, and Ryan Horne. “Appendix: Trac...",tracking-yu,https://doi.org/10.2307/j.ctv1vbd1d8.12.
7,2021,Maps and infographics in,Yellow River: A Natural and Unnatural History,NaN,"Maps and infographics in Mostern, Ruth. <i>The...",Map-info-yu,https://doi.org/10.2307/j.ctv1vbd1d8.12.
8,2021,Digital Tools and Ancient Empires: Using Netwo...,Journal of World History,NaN,"Horne, Ryan. “Digital Tools and Ancient Empire...",digital-tools-ancient-empires,https://muse.jhu.edu/article/794333
9,2020,Mapping Power: Using HGIS and Linked Open Data...,"Historical Geography, GIScience and Textual An...",This article illustrates how new advances in H...,"Horne, Ryan. “Mapping Power: Using HGIS and Li...",mapping-power,https://link.springer.com/chapter/10.1007/978-...


## Escape special characters

YAML is very picky about how it takes a valid string, so we are replacing single and double quotes (and ampersands) with their HTML encoded equivilents. This makes them look not so readable in raw format, but they are parsed and rendered nicely.

In [20]:
html_escape_table = {
    "&": "&amp;",
    '"': "&quot;",
    "'": "&apos;"
    }

def html_escape(text):
    """Produce entities within text."""
    return "".join(html_escape_table.get(c,c) for c in text)

## Creating the markdown files

This is where the heavy lifting is done. This loops through all the rows in the TSV dataframe, then starts to concatentate a big string (```md```) that contains the markdown for each type. It does the YAML metadata first, then does the description for the individual page.

In [21]:
import os
for row, item in publications.iterrows():
    
    md_filename = str(item.pub_date) + "-" + item.url_slug + ".md"
    html_filename = str(item.pub_date) + "-" + item.url_slug
    #year = item.pub_date[:4]
    year = item.pub_date

    ## YAML variables
    
    md = "---\ntitle: \""   + item.title + '"\n'
    
    md += """collection: publications"""
    
    md += """\npermalink: /publication/""" + html_filename
    
    if len(str(item.excerpt)) > 5:
        md += "\nexcerpt: '" + html_escape(item.excerpt) + "'"
    
    md += "\ndate: " + str(item.pub_date) 
    
    md += "\nvenue: '" + html_escape(item.venue) + "'"
    
    if len(str(item.paper_url)) > 5:
        md += "\npaperurl: '" + item.paper_url + "'"
    
    md += "\ncitation: '" + html_escape(item.citation) + "'"
    
    md += "\n---"
    
    ## Markdown description for individual page
        
    if len(str(item.excerpt)) > 5:
        md += "\n" + html_escape(item.excerpt) + "\n"
    
    if len(str(item.paper_url)) > 5:
        md += "\n[Download paper here](" + item.paper_url + ")\n" 
        
    md += "\nRecommended citation: " + item.citation
    
    md_filename = os.path.basename(md_filename)
       
    with open("../_publications/" + md_filename, 'w') as f:
        f.write(md)

These files are in the publications directory, one directory below where we're working from.

In [22]:
!ls ../_publications/

'ls' is not recognized as an internal or external command,
operable program or batch file.


In [23]:
!cat ../_publications/2009-10-01-paper-title-number-1.md

'cat' is not recognized as an internal or external command,
operable program or batch file.
